<style>
.info {border-left:5px solid #2563eb;background:#eff6ff;padding:12px 16px;margin:12px 0}
.task {border-left:5px solid #d97706;background:#fffbeb;padding:12px 16px;margin:12px 0}
.success {border-left:5px solid #059669;background:#ecfdf5;padding:12px 16px;margin:12px 0}
.warning {border-left:5px solid #dc2626;background:#fef2f2;padding:12px 16px;margin:12px 0}
.definition {border-left:5px solid #7c3aed;background:#f5f3ff;padding:12px 16px;margin:12px 0}
</style>

        # Python for Data Science
        ## TW13 · Session 2 · Case Study: Feature Engineering and Reproducible Preprocessing

        **Course level:** developing beginner  
        **Prior learning:** the cleaned TW13 case study and basic scikit-learn concepts  
        **Duration:** approximately 90 minutes  
        **Method:** explain → predict → run → change → practise

        Feature engineering converts valid source fields into useful representations. We protect against leakage by separating train and test data before learning imputation, scaling, or category rules.

        **Student name:** __________________________ &nbsp;&nbsp; **Date:** __________________


        ## Learning outcomes

        By the end of this session, you should be able to:

        - create date, value, and behavioural features
- distinguish predictors from identifiers and targets
- recognise target leakage
- build a ColumnTransformer for numeric and categorical data
- fit preprocessing on training data and validate transformed outputs


## Lesson plan

| Part | Time | Activity |
|---|---:|---|
| Retrieval | 10 min | Recall earlier Python and data skills |
| New concepts | 30 min | Learn with short, explained examples |
| Guided analysis | 20 min | Build one complete example with the lecturer |
| Student coding | 25 min | Complete five marked tasks |
| Review | 5 min | Knowledge check and exit ticket |

<div class="info"><strong>How to work:</strong> read the explanation, predict the result, run the cell with <code>Shift + Enter</code>, then change one thing. Edit only cells marked <strong>Student task</strong> unless your lecturer says otherwise.</div>


### Import and clean reproducibly

A function makes the transformation repeatable and testable.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

_candidates = [
    Path.cwd() / "data",
    Path.cwd() / "Python_Data_Science_TW09_TW17" / "data",
]
DATA_DIR = next((path for path in _candidates if path.exists()), _candidates[0])
if not DATA_DIR.exists():
    raise FileNotFoundError("Data folder not found. Start JupyterLab from the extracted course folder.")
print("Data folder:", DATA_DIR.resolve())

def clean_sales(source):
    frame = source.drop_duplicates().copy()
    frame["region"] = frame["region"].astype("string").str.strip().str.title().fillna("Unknown")
    frame["channel"] = frame["channel"].astype("string").str.strip().str.title().replace({"Telephone": "Phone"})
    frame["order_date"] = pd.to_datetime(
        frame["order_date"], format="mixed", dayfirst=True, errors="coerce"
    )
    frame["units"] = frame["units"].mask(~frame["units"].between(1, 20))
    frame["unit_price"] = frame["unit_price"].mask(frame["unit_price"] <= 0)
    frame["units"] = frame["units"].fillna(frame.groupby("category")["units"].transform("median"))
    frame["unit_price"] = frame["unit_price"].fillna(
        frame.groupby("category")["unit_price"].transform("median")
    )
    frame["satisfaction"] = frame["satisfaction"].fillna(frame["satisfaction"].median())
    return frame.dropna(subset=["order_date"]).copy()

raw = pd.read_csv(DATA_DIR / "business_sales_dirty.csv", encoding="utf-8-sig")
clean = clean_sales(raw)
print(clean.shape)


## 1. Feature engineering begins with meaning

A feature is a model input or analytical variable derived from available information. Good features are reproducible, interpretable, available at prediction time, and relevant to the question.


### Create transaction and date features

Every calculation uses fields available in the order row.


In [ ]:
features = clean.copy()
features["gross_revenue"] = features["units"] * features["unit_price"]
features["net_revenue"] = features["gross_revenue"] * (1 - features["discount"])
features["order_month"] = features["order_date"].dt.month
features["order_quarter"] = features["order_date"].dt.quarter
features["order_weekday"] = features["order_date"].dt.day_name()
features["is_weekend"] = features["order_date"].dt.dayofweek.ge(5).astype(int)
features["returned_flag"] = features["returned"].eq("Yes").astype(int)

print(features[[
    "gross_revenue", "net_revenue", "order_month",
    "order_quarter", "order_weekday", "is_weekend", "returned_flag"
]].head())


### Create a stable ratio with an explicit rule

Ratios need a defined denominator and protection against zero.


In [ ]:
features["revenue_per_marketing_pound"] = np.where(
    features["marketing_spend"] > 0,
    features["net_revenue"] / features["marketing_spend"],
    np.nan,
)
print(features["revenue_per_marketing_pound"].describe())


## 2. Identify roles: ID, predictor, target, or post-outcome

- Identifier: locates a record but usually should not be learned as a quantity.
- Predictor: information available when a prediction is made.
- Target: outcome to predict.
- Leakage: information that would not be available at prediction time or directly reveals the target.

If predicting whether an order will be returned at purchase time, a later return-processing code would be leakage.


### Create a role table

The role decision is tied to a defined prediction moment.


In [ ]:
feature_roles = pd.DataFrame({
    "column": [
        "order_id", "customer_id", "region", "channel", "units",
        "unit_price", "discount", "marketing_spend", "satisfaction",
        "returned_flag",
    ],
    "role": [
        "identifier", "identifier", "predictor", "predictor", "predictor",
        "predictor", "predictor", "predictor", "post-outcome/uncertain",
        "target",
    ],
})
print(feature_roles)


## 3. Split before learning preprocessing

The test set represents unseen data. Imputation values, means, standard deviations, and category vocabularies must be learned from training data only.


### Create predictors, target, and a stratified split

Satisfaction is omitted because it may be recorded after the outcome in this scenario.


In [ ]:
model_columns = [
    "region", "category", "channel", "units", "unit_price",
    "discount", "marketing_spend", "order_month",
    "order_quarter", "order_weekday", "is_weekend",
]
X = features[model_columns]
y = features["returned_flag"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Target rates:", round(y_train.mean(), 3), round(y_test.mean(), 3))


## 4. Build a preprocessing pipeline

Numerical features receive median imputation and standard scaling. Categorical features receive most-frequent imputation and one-hot encoding. handle_unknown='ignore' allows a category seen only in test data without crashing.


### Define numeric and categorical pipelines

The ColumnTransformer sends each named column group through the correct operations.


In [ ]:
numeric_features = [
    "units", "unit_price", "discount", "marketing_spend",
    "order_month", "order_quarter", "is_weekend",
]
categorical_features = ["region", "category", "channel", "order_weekday"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])
print(preprocessor)


### Fit on training data and transform both sets

Only fit_transform is used on training data. Test data receives transform only.


In [ ]:
X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)
output_names = preprocessor.get_feature_names_out()

print("Training matrix:", X_train_ready.shape)
print("Test matrix:", X_test_ready.shape)
print("First output names:", output_names[:10].tolist())


### Validate model-ready matrices

The matrices must contain the same feature columns, finite values, and unchanged row counts.


In [ ]:
assert X_train_ready.shape[0] == len(X_train)
assert X_test_ready.shape[0] == len(X_test)
assert X_train_ready.shape[1] == X_test_ready.shape[1]
assert np.isfinite(X_train_ready).all()
assert np.isfinite(X_test_ready).all()
assert len(output_names) == X_train_ready.shape[1]
print("Preprocessing validation passed.")


## 5. Feature quality review

More features are not automatically better. Review availability, stability, missingness, privacy, interpretability, redundancy, and possible bias. Document the prediction time and target definition.


### Create a compact feature manifest

The manifest communicates where each feature came from.


In [ ]:
feature_manifest = pd.DataFrame([
    {"feature": "units", "source": "order row", "transformation": "median impute + standardise"},
    {"feature": "region", "source": "order row", "transformation": "clean + one-hot"},
    {"feature": "order_month", "source": "order_date", "transformation": "calendar month + standardise"},
    {"feature": "is_weekend", "source": "order_date", "transformation": "day-of-week rule"},
    {"feature": "returned_flag", "source": "returned", "transformation": "binary target"},
])
print(feature_manifest)


## Guided output: prepare a model card input section

Before modelling, record the intended use, excluded uses, target, split, preprocessing, and major limitations.


### Create a machine-readable preparation summary

This summary will support TW14 model evaluation.


In [ ]:
preparation_summary = {
    "intended_use": "Teaching prediction of returned orders",
    "target": "returned_flag",
    "prediction_time": "At order placement",
    "training_rows": len(X_train),
    "test_rows": len(X_test),
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "known_limitations": [
        "Synthetic data",
        "Small sample",
        "Observational associations",
        "Target class imbalance",
    ],
}
print(preparation_summary)


## Student coding lab

Complete the tasks in order; later tasks depend on the documented feature roles.


        ### Student task 1: Engineer interpretable features

        <div class="task"><strong>Your job:</strong> Create gross revenue, net revenue, month, quarter, weekday, weekend flag, and returned flag. Validate ranges.</div>

        **Check your work**

        - Revenue formulas are correct.
- Calendar features come from parsed dates.
- Binary flags contain only 0 and 1.


In [ ]:
# STUDENT TASK 1
task_features = clean.copy()
# Add seven requested features and assertions.
print(task_features.head())


        ### Student task 2: Audit feature roles

        <div class="task"><strong>Your job:</strong> Label each candidate as identifier, predictor, target, post-outcome, or exclude. Add a reason and availability time.</div>

        **Check your work**

        - Every role has a reason.
- Identifiers are not treated as continuous measurements.
- Possible post-outcome leakage is recognised.


In [ ]:
# STUDENT TASK 2
role_audit = pd.DataFrame({
    "column": ["order_id", "customer_id", "region", "units", "satisfaction", "returned_flag"],
    "role": ["", "", "", "", "", ""],
    "reason": ["", "", "", "", "", ""],
    "available_when": ["", "", "", "", "", ""],
})
print(role_audit)


        ### Student task 3: Split correctly

        <div class="task"><strong>Your job:</strong> Build X and y from approved predictors and target, then create a 70/30 stratified split with random_state 123.</div>

        **Check your work**

        - Identifiers and target are absent from X.
- The split is stratified.
- Target rates are compared across sets.


In [ ]:
# STUDENT TASK 3
task_X = pd.DataFrame()
task_y = pd.Series(dtype=int)
# Add train_test_split.
print(task_X.shape, task_y.shape)


        ### Student task 4: Build preprocessing

        <div class="task"><strong>Your job:</strong> Create numeric and categorical pipelines, combine them in a ColumnTransformer, fit on training only, and transform test.</div>

        **Check your work**

        - Numerical values use median imputation and scaling.
- Categories use imputation and unknown-safe one-hot encoding.
- Test data is never passed to fit.


In [ ]:
# STUDENT TASK 4
task_preprocessor = None
# Define pipelines and transformations.
print(task_preprocessor)


        ### Student task 5: Validate and document output

        <div class="task"><strong>Your job:</strong> Assert row counts, equal feature widths, finite values, and feature-name agreement. Create a preparation summary.</div>

        **Check your work**

        - At least four programmatic checks are present.
- The record states target and prediction time.
- Leakage and class imbalance are listed as risks.



**Optional extension:** Place preprocessing before a simple classifier in one scikit-learn Pipeline.


In [ ]:
# STUDENT TASK 5
validation_results = {}
preparation_record = {}
print(validation_results)
print(preparation_record)


        ## Knowledge check

        Answer these without running new code first.

        1. What makes an engineered feature interpretable?
2. How does an identifier differ from a predictor?
3. What is target leakage?
4. Why split before fitting imputation or scaling?
5. What should a feature manifest record?


## Exit ticket and preparation

<div class="success"><strong>Exit ticket:</strong> write one idea you can explain, one operation you can code, and one question that remains.</div>

**Before the next session:** TW14 trains supervised classification and regression models using leakage-safe pipelines and evaluation metrics.
